# <center> Deliverable 2 by Fredrik and Jesper

Task 1)

In [ ]:


from pathlib import Path
import re
import json


def file_reader(filepath):
    """ 
    Reads the input file, 
    then translates the encodings from either UTF-8 or Latin_1.
    Returns file text content and encodings used
    """
    
    file = Path(filepath)
    # reading the file with the right encodings
    try:
        encoding="utf-8"
        text = file.read_text(encoding=encoding)
    except UnicodeDecodeError:
        encoding="latin-1"
        text = file.read_text(encoding=encoding)

    file_text_content = text
    encoding_used = encoding

    return (file_text_content, encoding_used)


# -----------------------------------------------------------

def log_parser(log_text):
    """
    Sorting the file by patterns using regex,
    and therefore removing all the white noise
    """

    pattern = r"\[(\S+ \S+)\] \[(\S+)\] ITEM=(\S+) PRICE=(\S+) (\S+) \((\S+)\)"
    data = []


    for line in log_text.splitlines():
        match = re.search(pattern, line)
        if match:
            data.append(match.groups())

    return data

Task 2)

In [ ]:


def find_log_files(directory_path):
    """
    Goes through every item in the directory, 
    and appends the item if it is a .log file.
    If an item is a directory,
    we resend it through the function.
    """
    
    base_dir = Path(directory_path)

    log_files = []

    for item in base_dir.iterdir():
        if item.is_file() and item.suffix == ".log":
            log_files.append(item)
        elif item.is_dir():
            sub_logs = find_log_files(item)
            log_files += sub_logs

    return log_files

# -----------------------------------------------------------

def active_registers(data, active_ids=None):
    """
    Sorts the nested data by dict or lists, 
    if the data in the dict is active we add the value,
    and resend it through the function.
    If the data is in a list we send it through the function again,
    to get to a deeper level.   
    
    """


    if active_ids is None:
        active_ids = set()


    if isinstance(data, dict):
        if data.get("active") is True and "id" in data:
            active_ids.add(data["id"])

        for value in data.values():
            active_registers(value, active_ids)


    elif isinstance(data, list):
        for item in data:
            active_registers(item, active_ids)


    return active_ids

    

Task 3)

In [ ]:
import numpy as np
import pandas as pd


# Task 3.1: Fjerning av pris-outliers

def remove_price_outliers(prices):

#Konverterer prisdata til et 1D NumPy-array og erstatter feilregistreringer

    # Konverterer til 1D NumPy float-array
    prices_array = np.array(prices, dtype=float)
    
    # Beregner gjennomsnitt (mu) og standardavvik (sigma)
    mean = np.mean(prices_array)
    std = np.std(prices_array)
    
    # Betingelse: |x - mu| > 3 * sigma
    outliers_mask = np.abs(prices_array - mean) > 3 * std
    
    # Beregner medianen til de gyldige prisene (der det IKKE er outlier)
    valid_prices = prices_array[~outliers_mask]
    valid_median = np.median(valid_prices)
    
    # Lager en kopi og erstatter outliers med medianen
    cleaned_prices = prices_array.copy()
    cleaned_prices[outliers_mask] = valid_median
    
    return cleaned_prices



# Task 3.2: Beregning av refusjonsrate

def compute_refund_ratio(sales_count: np.ndarray, refund_count: np.ndarray) -> np.ndarray:
    """
    Beregner elementvis refund ratio ved hjelp av NumPy array-operasjoner:
    Refund Ratio = Refund Count / (Sales Count + 1e-6)
    Legger til epsilon (1e-6) for å unngå 'division by zero'-feil.
    """
    epsilon = 1e-6
    refund_ratio = refund_count / (sales_count + epsilon)
    return refund_ratio

Task 4)

In [ ]:
# Task 4.1: Opprette DataFrame fra transaksjonsliste

def create_transaction_dataframe(transactions):
    """
    Konverterer en liste med transaksjons-dictionaries til en Pandas DataFrame,
    setter timestamp-kolonnen til datetime-format, setter den som indeks og
    sorterer DataFrame-en kronologisk.
    """
    df = pd.DataFrame(transactions)
    
    # Konverterer timestamp til datetime og setter som indeks
    df['timestamp'] = pd.to_datetime(df['timestamp'])
    df = df.set_index('timestamp')
    
    # Sorterer kronologisk
    df = df.sort_index()
    return df



# Task 4.2: Resampling av totalinntekt per time per vare

def calculate_hourly_revenue(df):
    """
    Grupperer data etter vare ('item') og resampler transaksjonene i 
    1-times tidsvinduer ('1h') for å beregne samlet inntekt (sum of price).
    """
    # Grupperer på 'item', resampler indeksen (timestamp) per time, og summerer prisen
    hourly_rev = df.groupby('item')['price'].resample('1h').sum().reset_index()
    return hourly_rev



# Task 4.3: Pivot-tabell over omsetning fordelt på status

def create_status_pivot_table(df):
    """
    Lager en pivot-tabell som viser samlet transaksjonsomsetning fordelt på
    vare ('item') langs radene og status (SUCCESS, REFUND, CANCELLED) langs kolonnene.
    """
    pivot = pd.pivot_table(
        df,
        values='price',
        index='item',
        columns='status',
        aggfunc='sum',
        fill_value=0.0
    )
    return pivot



# Task 4.4: Beregne refund ratio per vare med NumPy-funksjonen

def calculate_item_refund_ratios(df):
    """
    Beregner refusjonsrate per vare ved å telle antall vellykkede salg ('SUCCESS')
    og antall refusjoner ('REFUND'), og benytter compute_refund_ratio fra Task 3.2.
    """
    # Teller antall forekomster per item og status
    status_counts = df.groupby(['item', 'status']).size().unstack(fill_value=0)
    
    # Henter ut antall salg og refusjoner som NumPy-arrays
    sales_count = status_counts['SUCCESS'].to_numpy() if 'SUCCESS' in status_counts.columns else np.zeros(len(status_counts))
    refund_count = status_counts['REFUND'].to_numpy() if 'REFUND' in status_counts.columns else np.zeros(len(status_counts))
    
    # Kaller NumPy-funksjonen fra Task 3.2
    ratios = compute_refund_ratio(sales_count, refund_count)
    
    # Samler resultatene i en oversiktlig DataFrame
    refund_ratio_df = pd.DataFrame({
        'item': status_counts.index,
        'sales_count': sales_count,
        'refund_count': refund_count,
        'refund_ratio': ratios
    }).set_index('item')
    
    return refund_ratio_df

Task 5)